# 01 — Data Preparation
## Brand Vision — LogoDet-3K + Product-10K

Run this notebook **locally** (no GPU needed — file operations only).

### Prerequisites
- LogoDet-3K unzipped to `training/datasets/logodet3k/`
- Product-10K unzipped to `training/datasets/product10k/`

### Outputs
- `training/datasets/brand_logo_split/` — cropped logo images per brand (train/val/test)
- `training/datasets/product_category_split/` — product images per category (train/val/test)
- `training/datasets/yolo_logo_detection/` — YOLO-format detection dataset
- `training/datasets/yolo_logo_detection/logo_data.yaml` — YOLO dataset config

In [ ]:
# Cell 1 — Imports
import os
import json
import shutil
import random
from pathlib import Path
from PIL import Image

random.seed(42)

# Adjust this if running from a different working directory
NOTEBOOK_DIR = Path(os.getcwd())
DATASET_DIR  = NOTEBOOK_DIR / "datasets"

print(f"Dataset directory: {DATASET_DIR}")
print(f"Exists: {DATASET_DIR.exists()}")

In [ ]:
# Cell 2 — Inspect LogoDet-3K structure
logodet_dir = DATASET_DIR / "logodet3k"
assert logodet_dir.exists(), f"LogoDet-3K not found at {logodet_dir}"

all_brand_dirs = sorted([d for d in logodet_dir.iterdir() if d.is_dir()])
print(f"Total brand folders in LogoDet-3K: {len(all_brand_dirs)}")
print(f"Sample folder names: {[d.name for d in all_brand_dirs[:15]]}")

In [ ]:
# Cell 3 — Select target brands from the 50-class list
TARGET_BRANDS = [
    "Nike", "Adidas", "Apple", "Samsung", "Sony", "Puma", "Reebok",
    "NewBalance", "UnderArmour", "Levis", "HM", "Zara", "Gucci",
    "LouisVuitton", "Rolex", "RayBan", "Canon", "Nikon", "JBL", "Bose",
    "Dell", "HP", "Lenovo", "Asus", "Converse", "Vans", "Timberland",
    "TommyHilfiger", "CalvinKlein", "RalphLauren", "Lacoste", "Versace",
    "Armani", "Balenciaga", "TheNorthFace", "Columbia", "Patagonia",
    "Oakley", "Casio", "Fossil", "Huawei", "Xiaomi", "OnePlus",
    "Google", "Amazon", "Microsoft", "Intel",
]
TARGET_BRANDS = list(dict.fromkeys(TARGET_BRANDS))  # deduplicate

available_names = {d.name for d in all_brand_dirs}
matched = [b for b in TARGET_BRANDS if b in available_names]
unmatched = [b for b in TARGET_BRANDS if b not in available_names]

print(f"Matched {len(matched)}/{len(TARGET_BRANDS)} target brands")
print(f"Matched: {matched}")
if unmatched:
    print(f"Not found (check exact folder names): {unmatched}")

In [ ]:
# Cell 4 — Helper: crop logo regions from YOLO annotations
def crop_and_save_logos(
    brand_dir: Path,
    output_base: Path,
    brand_name: str,
    txt_files: list[Path],
    split: str,
) -> int:
    """
    Reads YOLO-format .txt annotations, crops the logo region from each image,
    and saves it to output_base/split/brand_name/.
    Returns count of saved crops.
    """
    dest = output_base / split / brand_name
    dest.mkdir(parents=True, exist_ok=True)
    saved = 0

    for txt_file in txt_files:
        img_file = brand_dir / (txt_file.stem + ".jpg")
        if not img_file.exists():
            img_file = brand_dir / (txt_file.stem + ".png")
        if not img_file.exists():
            continue

        try:
            img = Image.open(img_file).convert("RGB")
            w, h = img.size

            with open(txt_file) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue
                    _, cx, cy, bw, bh = map(float, parts)

                    x1 = int((cx - bw / 2) * w)
                    y1 = int((cy - bh / 2) * h)
                    x2 = int((cx + bw / 2) * w)
                    y2 = int((cy + bh / 2) * h)

                    # 10% padding around crop
                    px = int((x2 - x1) * 0.1)
                    py = int((y2 - y1) * 0.1)
                    crop = img.crop((
                        max(0, x1 - px), max(0, y1 - py),
                        min(w, x2 + px), min(h, y2 + py)
                    ))

                    if crop.width < 10 or crop.height < 10:
                        continue  # skip degenerate crops

                    out_name = f"{txt_file.stem}_{x1}_{y1}.jpg"
                    crop.save(dest / out_name, "JPEG", quality=90)
                    saved += 1
        except Exception as e:
            print(f"  ⚠ Error processing {img_file.name}: {e}")

    return saved

In [ ]:
# Cell 5 — Build brand classifier dataset (cropped logo images)
BRAND_SPLIT_DIR = DATASET_DIR / "brand_logo_split"
total_crops = 0

for brand in matched:
    brand_dir = logodet_dir / brand
    all_txts  = list(brand_dir.glob("*.txt"))
    random.shuffle(all_txts)

    n         = len(all_txts)
    train_end = int(n * 0.70)
    val_end   = int(n * 0.90)

    splits = {
        "train": all_txts[:train_end],
        "val":   all_txts[train_end:val_end],
        "test":  all_txts[val_end:],
    }

    brand_total = 0
    for split_name, files in splits.items():
        count = crop_and_save_logos(brand_dir, BRAND_SPLIT_DIR, brand, files, split_name)
        brand_total += count

    total_crops += brand_total
    print(f"  {brand}: {brand_total} crops saved")

print(f"\nBrand split done ✓ — {total_crops} total crops across {len(matched)} brands")

In [ ]:
# Cell 6 — Class count check (no class should have < 20 training samples)
print("Training sample counts per brand:")
low_count_brands = []

for brand_dir in sorted((BRAND_SPLIT_DIR / "train").iterdir()):
    count = len(list(brand_dir.glob("*.jpg")))
    status = "✓" if count >= 20 else "⚠ LOW"
    print(f"  {brand_dir.name}: {count} {status}")
    if count < 20:
        low_count_brands.append(brand_dir.name)

if low_count_brands:
    print(f"\n⚠ Brands with < 20 training samples: {low_count_brands}")
    print("Consider removing these from training or merging with similar brands.")
else:
    print("\n✅ All brands have ≥ 20 training samples")

In [ ]:
# Cell 7 — Build YOLO detection dataset (all logos = class 0)
YOLO_DET_DIR = DATASET_DIR / "yolo_logo_detection"

for split in ("train", "val"):
    (YOLO_DET_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (YOLO_DET_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

yolo_train, yolo_val = 0, 0

for brand in matched:
    brand_dir = logodet_dir / brand
    txt_files = list(brand_dir.glob("*.txt"))
    random.shuffle(txt_files)

    train_txts = txt_files[:int(len(txt_files) * 0.8)]
    val_txts   = txt_files[int(len(txt_files) * 0.8):]

    for split_name, files in [("train", train_txts), ("val", val_txts)]:
        for txt in files:
            img = brand_dir / (txt.stem + ".jpg")
            if not img.exists():
                img = brand_dir / (txt.stem + ".png")
            if not img.exists():
                continue

            # Copy image (prefix with brand name to avoid filename collisions)
            shutil.copy(img, YOLO_DET_DIR / "images" / split_name / f"{brand}_{img.name}")

            # Re-write label with class 0 for all brands (binary: logo vs no-logo)
            with open(txt) as f_in:
                lines = f_in.readlines()
            out_txt = YOLO_DET_DIR / "labels" / split_name / f"{brand}_{txt.name}"
            with open(out_txt, "w") as f_out:
                for line in lines:
                    parts = line.strip().split()
                    if parts:
                        f_out.write("0 " + " ".join(parts[1:]) + "\n")  # Force class_id=0

            if split_name == "train":
                yolo_train += 1
            else:
                yolo_val += 1

print(f"YOLO detection dataset built ✓")
print(f"  Train: {yolo_train} images")
print(f"  Val:   {yolo_val} images")

In [ ]:
# Cell 8 — Write logo_data.yaml (YOLO dataset config)
yolo_abs_path = str(YOLO_DET_DIR.resolve()).replace("\\", "/")

yaml_content = f"""# YOLOv8 logo detection dataset config
# Auto-generated by 01_data_preparation.ipynb

path: {yolo_abs_path}
train: images/train
val:   images/val

nc: 1          # 1 class only: 'logo' (brand identity is handled by EfficientNet)
names: ['logo']
"""

yaml_path = YOLO_DET_DIR / "logo_data.yaml"
yaml_path.write_text(yaml_content)
print(f"logo_data.yaml written → {yaml_path}")
print(yaml_content)

In [ ]:
# Cell 9 — Build product category dataset from Product-10K
product10k_dir = DATASET_DIR / "product10k" / "train"
assert product10k_dir.exists(), f"Product-10K not found at {product10k_dir}"

# Check category mapping file
cat_map_file = DATASET_DIR / "product10k" / "category.json"
if cat_map_file.exists():
    with open(cat_map_file) as f:
        cat_map = json.load(f)
    print(f"Category map sample (first 15):")
    for k, v in list(cat_map.items())[:15]:
        print(f"  {k}: {v}")
else:
    print("No category.json found — listing top-level folders instead:")
    subdirs = [d.name for d in sorted(product10k_dir.iterdir()) if d.is_dir()]
    print(f"  First 20: {subdirs[:20]}")

In [ ]:
# Cell 10 — Split product category dataset
# NOTE: After running Cell 9, update TARGET_CATEGORIES below to match
# the actual folder names found in your Product-10K download.

TARGET_CATEGORIES = [
    "shoes", "t-shirt", "jacket", "dress", "handbag", "watch", "smartphone",
    "laptop", "headphones", "sunglasses", "backpack", "jeans", "sneakers",
    "boots", "cap", "tablet", "camera", "bottle", "wallet", "belt",
]

CAT_SPLIT_DIR = DATASET_DIR / "product_category_split"
total_category_images = 0

for cat in TARGET_CATEGORIES:
    cat_dir = product10k_dir / cat
    if not cat_dir.exists():
        print(f"  ⚠ Category folder not found: {cat}")
        continue

    images = list(cat_dir.glob("*.jpg")) + list(cat_dir.glob("*.png"))
    random.shuffle(images)
    n = len(images)

    train_end = int(n * 0.70)
    val_end   = int(n * 0.90)

    for split_name, files in [
        ("train", images[:train_end]),
        ("val",   images[train_end:val_end]),
        ("test",  images[val_end:]),
    ]:
        dest = CAT_SPLIT_DIR / split_name / cat
        dest.mkdir(parents=True, exist_ok=True)
        for img_path in files:
            shutil.copy(img_path, dest / img_path.name)

    total_category_images += n
    print(f"  {cat}: {int(n*0.7)} train / {int(n*0.2)} val / {n - int(n*0.9)} test")

print(f"\nProduct category split done ✓ — {total_category_images} total images")

In [ ]:
# Cell 11 — Final summary
def count_images(base: Path) -> dict:
    counts = {}
    for split in ("train", "val", "test"):
        split_dir = base / split
        if split_dir.exists():
            counts[split] = sum(1 for _ in split_dir.rglob("*.jpg"))
    return counts

print("=" * 50)
print("PHASE 1 COMPLETE — Dataset Summary")
print("=" * 50)
print(f"\nBrand logo split:         {count_images(BRAND_SPLIT_DIR)}")
print(f"Product category split:   {count_images(CAT_SPLIT_DIR)}")
print(f"YOLO detection train:     {yolo_train} images")
print(f"YOLO detection val:       {yolo_val} images")
print(f"\nlogo_data.yaml:           {(YOLO_DET_DIR / 'logo_data.yaml').exists()}")
print("\n✅ Ready for Phase 2 — YOLOv8 training")